# Code 8.6.2: GPT-2 sizes on tiny Shakespeare

Computes the loss, perplexity, and bits per byte of GPT-2 small, medium, and large on the tiny Shakespeare validation split of Code 8.1.2, in non-overlapping windows of 1,024 tokens.


In [ ]:
import os
import urllib.request

# Tiny Shakespeare, the corpus Code 8.1.2 downloads as input.txt.
_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(_url, "input.txt")


In [ ]:
import math
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

text = open("input.txt", encoding="utf-8").read()       # tiny Shakespeare (Code 8.1.2)
tok = GPT2TokenizerFast.from_pretrained("gpt2")          # all GPT-2 sizes share this tokenizer
ids = torch.tensor(tok(text).input_ids)
val = ids[int(0.9 * len(ids)):]                          # the same validation split as Sections 1 and 4
n_bytes = len(tok.decode(val).encode("utf-8"))

@torch.no_grad()
def evaluate(name, n=1024):
    model = GPT2LMHeadModel.from_pretrained(name).eval()
    total, count = 0.0, 0
    for i in range(0, len(val) - 1, n):                  # non-overlapping windows of up to n tokens
        x = val[i: i + n + 1]
        logits = model(x[None, :-1]).logits[0]
        total += F.cross_entropy(logits, x[1:], reduction="sum").item()
        count += len(x) - 1
    return total / count, total / math.log(2) / n_bytes  # nats per token, bits per byte (Section 5.8)

print(len(val), "tokens,", n_bytes, "bytes")
for name in ["gpt2", "gpt2-medium", "gpt2-large"]:
    loss, bpb = evaluate(name)
    print(f"{name:12s} loss {loss:.3f}  perplexity {math.exp(loss):.1f}  bits per byte {bpb:.3f}")
# 33803 tokens, 104217 bytes
# gpt2         loss 4.009  perplexity 55.1  bits per byte 1.876
# gpt2-medium  loss 3.432  perplexity 31.0  bits per byte 1.606
# gpt2-large   loss 3.566  perplexity 35.4  bits per byte 1.668
